# 九州工業大学大学院生命体工学研究科 インターンシップ — 第2部 前半
## 層を深くすると何が起きるか

### 第2部前半の目的

第1部前半で、**層を1つ重ねるごとに参照できる範囲が1ホップずつ広がる**ことを確かめた。
Zachary のグラフでは3層でグラフ全体に届いていた。

ここから自然に出てくる予想は「層を深くすれば使える情報が増えて精度が上がる」である。
第1部前半の課題1a-5(3)で、この予想について自分の考えを書いてもらった。

**第2部前半では、その予想を実測で検証する。**
結果は多くの人の予想を裏切るはずである。そして重要なのは**その理由を突き止めること**である。

よくある説明に「層を深くすると**過平滑化**（over-smoothing、
すべての頂点の特徴が似た値に潰れてしまう現象）が起きるから精度が落ちる」というものがある。
この説明が本当に正しいのかどうかも、自分の手で確かめる。

### 到達目標
1. 層数と精度の関係を、誤差を込みで測定できる
2. 過平滑化が起きる条件を、学習を伴わない純粋な伝播で観測できる
3. 精度低下の原因が「汎化の失敗」なのか「最適化の失敗」なのかを、
   **訓練精度を見て切り分けられる**

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_dense_adj

def find_data_root():
    # 既定ではノートブックから見て ../data にデータを置く。
    # 手元で別の配置を試すときのために、候補を順に探す。
    # 環境変数 JIKKEN_DATA があればそれを最優先する。
    env = os.environ.get("JIKKEN_DATA")
    if env:
        return env
    for cand in ["../data", "../../data", "../../../data", "data"]:
        if (os.path.isdir(os.path.join(cand, "Planetoid"))
                or os.path.isdir(os.path.join(cand, "TUDataset"))):
            return cand
    return "../data"          # 見つからないときの既定値

DATA_ROOT = find_data_root()

# 本実習で扱うモデルは小さいため、計算スレッドを増やすとかえって遅くなる
# (並列化のオーバーヘッドが計算量を上回るため。実測で1.4〜1.5倍の差が出る)。
# 1スレッドに固定して実行時間を短縮する。
torch.set_num_threads(1)

dataset = Planetoid(root=f"{DATA_ROOT}/Planetoid", name="Cora")
data = dataset[0]
NUM_CLASSES = dataset.num_classes

print(f"Cora: 頂点数 {data.num_nodes}, 辺数 {data.num_edges}, "
      f"特徴次元 {dataset.num_features}, クラス数 {NUM_CLASSES}")

---
## 1. 学習抜きで「伝播だけ」を繰り返す

まず、**ニューラルネットワークの学習を一切せずに**、
第1部前半で作った $\hat{A}$ を特徴行列に何度も掛けたらどうなるかを見る。

$$X,\quad \hat{A}X,\quad \hat{A}^2X,\quad \hat{A}^3X,\quad \dots$$

これは「重みを恒等変換に固定したGNN」に相当する。
層を深くすることの**純粋な効果**だけを取り出して観察できる。

頂点の特徴が似てきたかどうかは、**頂点どうしの特徴ベクトルの平均コサイン類似度**で測る。
1に近づくほど「全頂点が同じ方向を向いた」＝潰れたことを意味する。

In [ ]:
A = to_dense_adj(data.edge_index, max_num_nodes=data.num_nodes)[0].numpy()
A_tilde = A + np.eye(len(A))
d = A_tilde.sum(axis=1)
A_hat = (d ** -0.5)[:, None] * A_tilde * (d ** -0.5)[None, :]

print("A_hat の形:", A_hat.shape)
print("対称か:", np.allclose(A_hat, A_hat.T))

> **課題2a-1** 次のセルの空欄を埋めて、平均コサイン類似度を計算する関数を完成させよ。

In [ ]:
def mean_cosine_similarity(H, n_sample=500, seed=0):
    # 頂点をランダムに n_sample 個選び、その全ペアのコサイン類似度の平均を返す
    rng = np.random.default_rng(seed)
    idx = rng.choice(H.shape[0], size=min(n_sample, H.shape[0]), replace=False)
    Z = H[idx]

    # ----- ここから: 各行を長さ1に正規化する -----
    # ヒント: np.linalg.norm(Z, axis=1, keepdims=True) が各行の長さ
    #         0除算を避けるため分母に 1e-12 を足すこと
    Z =
    # ----- ここまで -----

    S = Z @ Z.T                              # 全ペアのコサイン類似度
    iu = np.triu_indices(len(idx), k=1)      # 対角と重複を除く
    return float(S[iu].mean())

print("定義した")

In [ ]:
# --- 既知の入力で検算する ---
# 全ての頂点が同じ方向を向いていれば、平均コサイン類似度は 1 になる
_same = np.ones((4, 3))
assert abs(mean_cosine_similarity(_same, n_sample=4) - 1.0) < 1e-9, \
    "同じ向きのベクトルばかりなら 1 になるはず"

# 3頂点のうち2つが同方向、1つが直交している場合
#   ペアは (0,1)=1, (0,2)=0, (1,2)=0 の3つ -> 平均 1/3
_mix = np.array([[1.0, 0.0],
                 [1.0, 0.0],
                 [0.0, 1.0]])
assert abs(mean_cosine_similarity(_mix, n_sample=3) - 1/3) < 1e-9, \
    "正規化のしかたを見直すこと (各行を長さ1にしてから内積を取る)"

# 長さだけが違うベクトルは、向きが同じなら類似度1になる (正規化が効いている確認)
_scaled = np.array([[1.0, 0.0], [5.0, 0.0], [100.0, 0.0]])
assert abs(mean_cosine_similarity(_scaled, n_sample=3) - 1.0) < 1e-9, \
    "長さの違いが結果に影響している。正規化ができていない"

print("検算OK")

In [ ]:
H = data.x.numpy().astype(np.float64)

print(f"{'k (伝播回数)':>12} | {'平均コサイン類似度':>18}")
print("-" * 34)
ks, sims = [], []
for k in range(0, 65):
    if k > 0:
        H = A_hat @ H
    if k in (0, 1, 2, 4, 8, 16, 32, 64):
        s = mean_cosine_similarity(H)
        ks.append(k); sims.append(s)
        print(f"{k:>12} | {s:>18.6f}")

In [ ]:
import matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans"]
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ks, sims, marker="o")
ax.set_xscale("symlog", base=2)
ax.set_xlabel("number of propagations  k")
ax.set_ylabel("mean cosine similarity")
ax.set_title("Over-smoothing by repeated propagation (no training)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**観察**: 伝播を繰り返すほど頂点どうしの特徴が似ていく。これが**過平滑化**である。
学習を伴わない純粋な伝播では、確かに過平滑化が起きることが確認できた。

ここで早合点しないこと。
確認できたのは「**重みを学習しなければ**過平滑化が起きる」ことだけである。
実際のGNNは重みを学習する。次節で、学習ありの場合に何が起きるかを見る。

---
## 2. 学習ありのGCNで層数を変える

層数 $L$ を変えて Cora の頂点分類を学習させ、精度を測る。

**重要**: テスト精度だけでなく、**訓練精度と訓練損失も必ず記録する**。
これが原因の切り分けに決定的に効く。

- 訓練精度が高い（100%近い）のにテスト精度が低い → 学習はできている。**汎化に失敗**している
- 訓練精度も低い → そもそも**学習(最適化)に失敗**している

この2つは原因も対策もまったく違う。混同してはいけない。

In [ ]:
class DeepGCN(torch.nn.Module):
    # GCNConv を L 層重ねたモデル。residual=True で残差接続を入れる
    def __init__(self, num_layers, in_dim, hid, out_dim, residual=False):
        super().__init__()
        self.residual = residual
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hid))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hid, hid))
        self.lin = Linear(hid, out_dim)

    def forward(self, x, edge_index, return_hidden=False):
        for i, conv in enumerate(self.convs):
            out = F.relu(conv(x, edge_index))
            # 2層目以降は入力を足し込む (残差接続)
            x = x + out if (self.residual and i > 0) else out
        h = x
        return (self.lin(h), h) if return_hidden else self.lin(h)


def run(num_layers, seed, residual=False, epochs=200, hid=16):
    torch.manual_seed(seed)
    model = DeepGCN(num_layers, dataset.num_features, hid, NUM_CLASSES, residual)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        out, h = model(data.x, data.edge_index, return_hidden=True)
        pred = out.argmax(dim=1)
        tr = float((pred[data.train_mask] == data.y[data.train_mask]).float().mean())
        te = float((pred[data.test_mask]  == data.y[data.test_mask] ).float().mean())
    return tr, te, float(loss.detach()), mean_cosine_similarity(h.numpy())

print("定義した")

> **課題2a-2** 次のセルを実行し、層数を変えたときの挙動を記録せよ。数分かかる。

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
LAYERS = [2, 3, 4, 6, 8]
plain = {}

print(f"{'層数':>4} | {'テスト精度(平均±標準偏差)':<22} | {'訓練精度':>8} | "
      f"{'訓練損失':>8} | {'cos類似度':>9} | 時間")
print("-" * 82)

for L in LAYERS:
    t0 = time.time()
    r = np.array([run(L, s) for s in SEEDS])
    plain[L] = r
    print(f"{L:>4} | {r[:,1].mean()*100:6.2f} % ± {r[:,1].std()*100:5.2f}        | "
          f"{r[:,0].mean()*100:7.1f}% | {r[:,2].mean():8.4f} | "
          f"{r[:,3].mean():9.4f} | {time.time()-t0:5.1f}s")

### 結果を読む

上の表を、次の順序で読むこと。

1. **テスト精度**は層数とともにどうなったか。第1部前半の予想は当たっていたか。
2. **訓練精度**を見る。層数が浅いうちは訓練精度はどうなっているか。
   訓練精度が高いままテスト精度だけ落ちているなら、原因は汎化の失敗である。
3. 最も深い設定で**訓練精度と訓練損失**はどうなったか。
   ここで訓練精度が落ちているなら、**訓練データにすら当てはめられていない**ということである。
   その場合、テスト精度の低下を「汎化の問題」だけで説明することはできない。
   （ただし「だから原因は勾配である」と断定することもできない。
   言えるのは「学習そのものがうまくいっていない」ところまでである。）
4. **cos類似度**の列を見る。第1節では伝播のたびに単調に1へ近づいた。
   学習ありの場合も同じように単調に増えているか。
   もし増えていないなら、「過平滑化が進んだから精度が落ちた」という筋書きは
   **この実験結果からは読み取れない**ことになる。
   （「過平滑化は起きていない」と証明できるわけではない。
   手元のデータがその説明を支持していない、というだけである。
   実験結果から言えることと言えないことを区別すること。）

5. **標準偏差**の列を見る。層が深くなるとばらつきはどうなったか。
   平均だけでなくばらつきの変化も、何が起きているかの手がかりになる。

ここで読み取ったことは、最後の記述課題（課題2a-6）でまとめて答える。
表を見ながらメモを取っておくこと。

---
## 3. その崩れはどれくらい確実な現象か

第2節で、最も深い設定では訓練精度そのものが落ちることを観察した。
しかしそれは、たった5通りのseedで見ただけである。

**「深くすると訓練に失敗する」と言い切ってよいのか。**

seedを10通りに増やして、8層の訓練精度が実行ごとにどうばらつくかを確かめる。

> **課題2a-3** 次のセルを実行し、8層の訓練精度がseedによってどう変わるかを記録せよ。
> 安定している6層と比較すること。1分ほどかかる。

In [ ]:
print(f"{'seed':>5} | {'6層 訓練':>9} | {'6層 テスト':>10} | {'8層 訓練':>9} | {'8層 テスト':>10}")
print("-" * 56)

rows6, rows8 = [], []
for s_ in range(10):
    t6, t8 = run(6, s_), run(8, s_)
    rows6.append(t6); rows8.append(t8)
    print(f"{s_:>5} | {t6[0]*100:8.1f}% | {t6[1]*100:9.2f}% | "
          f"{t8[0]*100:8.1f}% | {t8[1]*100:9.2f}%")

a6, a8 = np.array(rows6), np.array(rows8)
print("-" * 56)
print(f"{'平均':>5} | {a6[:,0].mean()*100:8.1f}% | {a6[:,1].mean()*100:9.2f}% | "
      f"{a8[:,0].mean()*100:8.1f}% | {a8[:,1].mean()*100:9.2f}%")
print()
print(f"6層の訓練精度: {a6[:,0].min()*100:.1f}% 〜 {a6[:,0].max()*100:.1f}%")
print(f"8層の訓練精度: {a8[:,0].min()*100:.1f}% 〜 {a8[:,0].max()*100:.1f}%")
print(f"8層で訓練精度が95%以上に達した回数: {int((a8[:,0] >= 0.95).sum())} / 10")

**ここが第2部前半で最も重要な観察である。**

8層の訓練精度は、seedによって大きく変わったはずである。
つまり「8層にすると訓練に失敗する」のではなく、
**「8層にすると訓練が成功したり失敗したりする」**というのが正しい記述である。

第2節で見た低い平均値は、失敗した回に引きずられた平均であって、
「毎回失敗する」という意味ではない。

ここで第2節の**テスト精度の標準偏差**を見直してほしい。
層数とともに標準偏差が大きくなっていたはずである。
**あのばらつきの大きさ自体が、この不安定さの現れだった。**
平均だけを見ていては気づけない。

実験結果を報告するときは「深くすると訓練に失敗する」と断定せず、
**「深くすると訓練が不安定になり、失敗することがある」**と書くこと。
実験から言えることと言えないことを区別するのが、本実習で身につけてほしい態度である。

---
## 4. 残差接続を入れる

第3節で、深い設定では**訓練そのものが不安定になる**ことが分かった。
訓練データにすら当てはめられない回があるのだから、
まず手を打つべきは「学習が安定して進むようにすること」である。

その代表的な手法が**残差接続** (residual connection) である。
これは深いネットワークで勾配が届きにくくなる問題への対策として提案されたものだが、
**効いたからといって原因が勾配だけだったと決まるわけではない**。
効果があったという事実と、その理由の説明は別のものである。

残差接続では、各層の出力をそのまま次へ渡すのではなく、入力を足して渡す。

$$\bm{x}^{(l+1)} = \bm{x}^{(l)} + \mathrm{ReLU}\!\left(\mathrm{Conv}(\bm{x}^{(l)})\right)$$

`DeepGCN` は `residual=True` でこれを有効にしてある。

> **課題2a-4** 次のセルを実行し、残差接続の有無を比較せよ。
> テスト精度だけでなく**訓練精度**が回復しているかに注目すること。

In [ ]:
print(f"{'層数':>4} | {'残差なし':>22} | {'残差あり':>22} | {'改善':>8}")
print("-" * 68)

for L in [4, 6, 8]:
    r = np.array([run(L, s, residual=True) for s in SEEDS])
    base = plain[L]
    gain = (r[:,1].mean() - base[:,1].mean()) * 100
    print(f"{L:>4} | {base[:,1].mean()*100:6.2f} % ± {base[:,1].std()*100:5.2f} "
          f"(訓練{base[:,0].mean()*100:5.1f}%) | "
          f"{r[:,1].mean()*100:6.2f} % ± {r[:,1].std()*100:5.2f} "
          f"(訓練{r[:,0].mean()*100:5.1f}%) | {gain:+7.2f}pt")

**観察のしかた**: 改善の大きさが層数によってどう違うかを見ること。
また、残差ありのときに**訓練精度**がどう変わったかに注目すること。
第3節で見た不安定さが解消されているかが要点である。

---
## 5. 第2部前半の課題

- **課題2a-1** 平均コサイン類似度の実装（第1節）
- **課題2a-2** 層数を変えた測定（第2節）
- **課題2a-3** 8層の結果がseedにどれだけ依存するかの確認（第3節）
- **課題2a-4** 残差接続の効果の測定（第4節）
- **課題2a-5** 層数12と16でも測定し、表に追加せよ。残差の有無の両方で行うこと。
- **課題2a-6** 次のセルに答えを記述せよ。

In [ ]:
# 課題2a-5 をここに書く

### 課題2a-6（記述）

**(1)** テスト精度は層数とともにどう変化したか。
第1部前半の課題1a-5(3)で書いた予想は当たっていたか。

（ここに記述）

**(2)** 訓練精度の列から何が読み取れるか。浅い層と深い層で何が違うか。
「汎化の失敗」と「訓練そのものの失敗」という言葉を使って説明せよ。

（ここに記述）

**(3)** 第1節（学習なしの伝播）と第2節（学習あり）で、
cos類似度の振る舞いはどう違ったか。
「過平滑化のせいで精度が落ちた」という説明は、あなたの実験結果から読み取れるか。
**読み取れない場合、それは「過平滑化が起きていない」ことの証明になるか**も述べよ。

（ここに記述）

**(4)** 第3節でseedを10通りに増やした結果、8層について何が分かったか。
5通りだけを見て書いたであろう結論と、どこが変わったか。

（ここに記述）

**(5)** 残差接続はどの層数で最も効果が大きかったか。訓練精度の変化を根拠に述べよ。
また、残差接続が効いたことから「原因は勾配が届かないことだった」と結論してよいか。
理由とともに述べよ。

（ここに記述）

**(6)** 以上を踏まえて、「GNNの層数は何層にすべきか」という問いに、
あなたの実験結果に基づいて答えよ。単一の数値ではなく、条件付きで答えてよい。

（ここに記述）